In [1]:
!nvidia-smi

Sat Oct 24 07:44:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U transformers datasets accelerate evaluate scikit-learn

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [3]:
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split

ds = load_dataset("mteb/banking77")
train_full = ds["train"].to_pandas()
test_df = ds["test"].to_pandas()

# label names, cleaned
label_map = (train_full[["label", "label_text"]].drop_duplicates()
             .sort_values("label"))
label_names = [n.replace("?", "") for n in label_map["label_text"]]
id2label = {i: n for i, n in enumerate(label_names)}
label2id = {n: i for i, n in id2label.items()}
print(len(label_names), "classes")

# validation split (10%), stratified so every class appears in both
train_df, val_df = train_test_split(
    train_full, test_size=0.1, stratify=train_full["label"], random_state=42
)
print(len(train_df), len(val_df), len(test_df))

cols = ["text", "label"]
train_ds = Dataset.from_pandas(train_df[cols].reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df[cols].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[cols].reset_index(drop=True))

README.md:   0%|          | 0.00/16.6k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  294kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 91.4kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9993 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3076 [00:00<?, ? examples/s]

77 classes
8993 1000 3076


In [4]:
from transformers import AutoTokenizer, DataCollatorWithPadding

MODEL = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=64)

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
val_tok = val_ds.map(tokenize, batched=True, remove_columns=["text"])
test_tok = test_ds.map(tokenize, batched=True, remove_columns=["text"])

collator = DataCollatorWithPadding(tokenizer=tokenizer)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/8993 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3076 [00:00<?, ? examples/s]

In [5]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=77, id2label=id2label, label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="distilbert-banking77",
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    save_total_limit=1,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,1.307111,0.802000,0.756514
2,2.036507,0.534150,0.879000,0.869517
3,2.036507,0.364262,0.911000,0.906654
4,0.319742,0.319112,0.909000,0.907217
5,0.319742,0.309173,0.912000,0.910468


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1410, training_loss=0.8724357280325382, metrics={'train_runtime': 189.5449, 'train_samples_per_second': 237.226, 'train_steps_per_second': 7.439, 'total_flos': 534969047755224.0, 'train_loss': 0.8724357280325382, 'epoch': 5.0})

In [7]:
out = trainer.predict(test_tok)
preds = np.argmax(out.predictions, axis=-1)
y_true = out.label_ids

acc = accuracy_score(y_true, preds)
f1 = f1_score(y_true, preds, average="macro")
print(f"Test Accuracy: {acc:.4f} | Macro-F1: {f1:.4f}")

Test Accuracy: 0.9217 | Macro-F1: 0.9217


In [8]:
import pandas as pd
from sklearn.metrics import classification_report

report = classification_report(
    y_true, preds, target_names=label_names, output_dict=True
)
rep = pd.DataFrame(report).T.iloc[:77]

tfidf_weak = [
    "balance_not_updated_after_bank_transfer", "top_up_failed",
    "pending_transfer", "card_not_working",
    "wrong_exchange_rate_for_cash_withdrawal", "card_payment_not_recognised",
    "verify_my_identity", "transfer_not_received_by_recipient",
    "topping_up_by_card", "top_up_reverted",
]
print("Transformer F1 on the classes TF-IDF found hardest:")
print(rep.loc[tfidf_weak, ["precision", "recall", "f1-score"]].round(3))

print("\nTransformer's own 10 weakest classes:")
print(rep.sort_values("f1-score").head(10)[["precision", "recall", "f1-score"]].round(3))

Transformer F1 on the classes TF-IDF found hardest:
                                         precision  recall  f1-score
balance_not_updated_after_bank_transfer      0.825   0.825     0.825
top_up_failed                                0.900   0.900     0.900
pending_transfer                             0.882   0.769     0.822
card_not_working                             0.860   0.925     0.892
wrong_exchange_rate_for_cash_withdrawal      0.944   0.850     0.895
card_payment_not_recognised                  0.875   0.875     0.875
verify_my_identity                           0.818   0.900     0.857
transfer_not_received_by_recipient           0.755   0.949     0.841
topping_up_by_card                           0.892   0.825     0.857
top_up_reverted                              0.919   0.850     0.883

Transformer's own 10 weakest classes:
                                         precision  recall  f1-score
reverted_card_payment                        0.791   0.850     0.819
pending_tran

In [9]:
import json
results = {"model": MODEL, "dataset": "mteb/banking77",
           "test_accuracy": float(acc), "test_macro_f1": float(f1)}
with open("results_distilbert.json", "w") as f:
    json.dump(results, f, indent=2)

from google.colab import drive
drive.mount("/content/drive")
trainer.save_model("/content/drive/MyDrive/banking77_distilbert_model")
tokenizer.save_pretrained("/content/drive/MyDrive/banking77_distilbert_model")

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/banking77_distilbert_model/tokenizer_config.json',
 '/content/drive/MyDrive/banking77_distilbert_model/tokenizer.json')